# Chapter 10 — State Pattern
## *The State of Things*

**Intent:** Allow an object to alter its behavior when its internal state changes.  
The object will appear to change its class.

### Relationship to Strategy
State and Strategy share the same class diagram, but different **intent**:
- **Strategy:** Client chooses the algorithm; it's a composition of behavior.
- **State:** State objects drive transitions; the context rarely knows which state it's in.

### Book context
A gumball machine: states are `NoQuarter`, `HasQuarter`, `Sold`, `SoldOut`.  
Without the pattern: a large `if/elif` block grows with every new state.  
With the pattern: each state is its own class.

In [ ]:
from __future__ import annotations
from abc import ABC, abstractmethod

class State(ABC):
    @abstractmethod
    def insert_quarter(self): ...
    @abstractmethod
    def eject_quarter(self): ...
    @abstractmethod
    def turn_crank(self): ...
    @abstractmethod
    def dispense(self): ...

In [ ]:
class GumballMachine:
    def __init__(self, count: int):
        self._count = count
        # Concrete states hold a reference back to the machine
        self.sold_out    = SoldOutState(self)
        self.no_quarter  = NoQuarterState(self)
        self.has_quarter = HasQuarterState(self)
        self.sold        = SoldState(self)

        self._state: State = self.sold_out if count == 0 else self.no_quarter

    # Delegate actions to current state
    def insert_quarter(self): self._state.insert_quarter()
    def eject_quarter(self):  self._state.eject_quarter()
    def turn_crank(self):
        self._state.turn_crank()
        self._state.dispense()

    def set_state(self, s: State): self._state = s

    def release_ball(self):
        if self._count > 0:
            print("A gumball comes rolling out...")
            self._count -= 1

    @property
    def count(self): return self._count

    def __repr__(self):
        return (f"GumballMachine [gumballs={self._count}, "
                f"state={type(self._state).__name__}]")

In [ ]:
class NoQuarterState(State):
    def __init__(self, machine: GumballMachine): self._m = machine

    def insert_quarter(self):
        print("You inserted a quarter.")
        self._m.set_state(self._m.has_quarter)

    def eject_quarter(self):  print("You haven't inserted a quarter.")
    def turn_crank(self):     print("You turned, but there's no quarter.")
    def dispense(self):       print("You need to pay first.")


class HasQuarterState(State):
    def __init__(self, machine: GumballMachine): self._m = machine

    def insert_quarter(self): print("You can't insert another quarter.")

    def eject_quarter(self):
        print("Quarter returned.")
        self._m.set_state(self._m.no_quarter)

    def turn_crank(self):
        print("You turned...")
        self._m.set_state(self._m.sold)

    def dispense(self): print("No gumball dispensed.")


class SoldState(State):
    def __init__(self, machine: GumballMachine): self._m = machine

    def insert_quarter(self): print("Please wait — dispensing gumball.")
    def eject_quarter(self):  print("Sorry, you already turned the crank.")
    def turn_crank(self):     print("Turning twice doesn't get you another.")

    def dispense(self):
        self._m.release_ball()
        if self._m.count == 0:
            print("Oops, out of gumballs!")
            self._m.set_state(self._m.sold_out)
        else:
            self._m.set_state(self._m.no_quarter)


class SoldOutState(State):
    def __init__(self, machine: GumballMachine): self._m = machine

    def insert_quarter(self): print("Machine sold out — coin returned.")
    def eject_quarter(self):  print("You haven't inserted a quarter.")
    def turn_crank(self):     print("Machine sold out.")
    def dispense(self):       print("No gumball dispensed.")

In [ ]:
# ── Demo ─────────────────────────────────────────────────────────────────────
machine = GumballMachine(2)
print(machine, "\n")

machine.insert_quarter()
machine.turn_crank()
print(machine, "\n")

machine.insert_quarter()
machine.turn_crank()
print(machine, "\n")

print("--- machine sold out ---")
machine.insert_quarter()
machine.turn_crank()

## Python `enum`-based FSM (lighter approach)

In [ ]:
from enum import Enum, auto

class TrafficLight:
    class Color(Enum):
        RED    = auto()
        YELLOW = auto()
        GREEN  = auto()

    _transitions = {
        Color.RED:    Color.GREEN,
        Color.GREEN:  Color.YELLOW,
        Color.YELLOW: Color.RED,
    }

    def __init__(self):
        self._state = self.Color.RED

    def tick(self):
        self._state = self._transitions[self._state]
        print(f"Light → {self._state.name}")


light = TrafficLight()
for _ in range(6):
    light.tick()

## Interview cheat-sheet

| Question | Answer |
|---|---|
| Problem solved? | Eliminates large conditional logic (if/elif chains) based on state. |
| State vs Strategy? | State: transitions are internal, states know about each other. Strategy: client picks the algorithm explicitly. |
| Who drives transitions? | Usually the State objects themselves set the next state on the context. |
| Real-world uses? | UI component states, TCP connection, vending machines, game characters, parsers. |

**Pattern family:** Behavioral